In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import ddo_stats as stats
import logging

logger = logging.getLogger(__name__)

def configure_logging():
    logging.basicConfig(filename='ddobuilder-comparer.log', level=logging.DEBUG)
    logging.info("Start ------------------------------------------------------------------------------------")

if __name__ == '__main__':
    configure_logging()

In [2]:
ddo_file_from_pandas = pd.read_fwf("ddo_files_from_maetrim_builder/paste_your_export_here.txt", colspecs=[(0, 999)], header=None)

# To iterate, we convert to a NumPy array, then to a standard Python list. Every row in the text file will be an element in the list.
ddo_file_in_numpy = ddo_file_from_pandas.to_numpy()
ddo_file = []

for row in ddo_file_in_numpy:
    ddo_file.append(str(row[0]))

In [3]:
melee_power = stats.get_stat(ddo_file, stats.labels["melee_power"])
doublestrike = stats.get_stat(ddo_file, stats.labels["doublestrike"])
helpless_damage_bonus = stats.get_stat(ddo_file, stats.labels["helpless_damage_bonus"])
expected_damage = stats.get_expected_damage(ddo_file)
# TODO: add sneak attack dice and bonus damage
offensive_fitness_score = (expected_damage *
                            stats.convert_to_factor(melee_power) * 
                            stats.convert_to_factor(doublestrike) * 
                            stats.convert_to_factor(helpless_damage_bonus))

In [4]:
hit_points = stats.get_stat(ddo_file, stats.labels["hit_points"])
prr = stats.get_stat(ddo_file, stats.labels["prr"])
prr_percentage = stats.reduction_rating_to_percentage(prr)
logger.debug(f'The physical damage percentage that\'s reduced by PRR: {prr_percentage}.')
mrr = stats.get_stat(ddo_file, stats.labels["mrr"])
mrr_percentage = stats.reduction_rating_to_percentage(mrr)
logger.debug(f'The elemental damage percentage that\'s reduced by MRR: {mrr_percentage}.')
dodge = stats.get_stat(ddo_file, stats.labels["dodge"])
armor_class = stats.get_stat(ddo_file, stats.labels["armor_class"])
defensive_fitness_score = (hit_points * 
                           (1.0 + prr_percentage) *
                           (1.0 + mrr_percentage) *
                           (1.0 + stats.normalize_dodge(dodge)) *
                           (1.0 + stats.normalize_armor_class(armor_class)))


In [5]:

logger.info(f'An offensive fitness score for main hand items only: {offensive_fitness_score:.0f}.')
logger.info(f'A defensive fitness score (A bit like \'Effective Hit Points\'): {defensive_fitness_score:.0f}.')